Data Pipelines & Automation

In [51]:
import requests
import pandas as pd
from datetime import datetime
import sqlite3
from pathlib import Path

In [73]:
pip install python-dotenv

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [74]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



  Using cached pip-26.2.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.2.1-py3-none-any.whl (1.8 MB)


In [75]:
pip install requests

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



In [76]:
pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeableNote: you may need to restart the kernel to use updated packages.



ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [ ]:
API_KEY = "YOUR_API_KEY_HERE"

**Define locations**

In [26]:
cities = [
    {
        "City": "Lagos",
        "State": "Lagos",
        "Region": "South West"
    },
    {
        "City": "Ibadan",
        "State": "Oyo",
        "Region": "South West"
    },
    {
        "City": "Akure",
        "State": "Ondo",
        "Region": "South West"
    },
    {
        "City": "Enugu",
        "State": "Enugu",
        "Region": "South East"
    },
    {
        "City": "Umuahia",
        "State": "Abia",
        "Region": "South East"
    },
    {
        "City": "Awka",
        "State": "Anambra",
        "Region": "South East"
    },
    {
        "City": "Port Harcourt",
        "State": "Rivers",
        "Region": "South South"
    },
    {
        "City": "Calabar",
        "State": "Cross River",
        "Region": "South South"
    },
    {
        "City": "Abuja",
        "State": "FCT",
        "Region": "North Central"
    },
    {
        "City": "Jos",
        "State": "Plateau",
        "Region": "North Central"
    },
    {
        "City": "Kano",
        "State": "Kano",
        "Region": "North West"
    },
    {
        "City": "Maiduguri",
        "State": "Borno",
        "Region": "North East"
    }
]

test on one location first

In [9]:
city = "Lagos"
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [71]:
# the raw API response
data = response.json()
data

{'coord': {'lon': 3.75, 'lat': 6.5833},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04d'}],
 'base': 'stations',
 'main': {'temp': 27.06,
  'feels_like': 29.4,
  'temp_min': 27.06,
  'temp_max': 27.06,
  'pressure': 1012,
  'humidity': 75,
  'sea_level': 1012,
  'grnd_level': 1012},
 'visibility': 10000,
 'wind': {'speed': 2.69, 'deg': 224, 'gust': 3.7},
 'clouds': {'all': 100},
 'dt': 1787063950,
 'sys': {'country': 'NG', 'sunrise': 1787031567, 'sunset': 1787075920},
 'timezone': 3600,
 'id': 2332453,
 'name': 'Lagos',
 'cod': 200}

In [11]:
#see the top-level fields
data.keys()

dict_keys(['coord', 'weather', 'base', 'main', 'visibility', 'wind', 'clouds', 'dt', 'sys', 'timezone', 'id', 'name', 'cod'])

In [12]:
#Look at the weather section
print(data["weather"])
print(data["weather"][0]["main"])
print(data["weather"][0]["description"])

[{'id': 804, 'main': 'Clouds', 'description': 'overcast clouds', 'icon': '04d'}]
Clouds
overcast clouds


In [13]:
#Look at the main weather measurements
print(data["main"])
#get temperature
print(data["main"]["temp"])
#gets feels-like temperature
print(data["main"]["feels_like"])
#get humidity
data["main"]["humidity"]

{'temp': 27.06, 'feels_like': 29.4, 'temp_min': 27.06, 'temp_max': 27.06, 'pressure': 1012, 'humidity': 75, 'sea_level': 1012, 'grnd_level': 1012}
27.06
29.4


75

In [14]:
#Look at wind
print(data["wind"])
#get windspeed
print(data["wind"]["speed"])
#wind direction
print(data["wind"]["deg"])

{'speed': 2.69, 'deg': 224, 'gust': 3.7}
2.69
224


In [15]:
#Look at location
print(data["coord"])
#Look at the city
print(data["name"])
#Look at cloudiness
print(data["clouds"])
#cloudiness percentage
print(data["clouds"]["all"])

{'lon': 3.75, 'lat': 6.5833}
Lagos
{'all': 100}
100


In [16]:
#Look at visibility
print(data["visibility"])

10000


In [17]:
#Check the timezone information
print(data["timezone"])
#1 hour × 60 × 60 = 3600 seconds

3600


In [18]:
#Look at the raw timestamp
print(data["dt"])

1787063950


In [9]:
weather_data = {
    "City": data["name"],
    "Temperature_C": data["main"]["temp"],
    "Humidity_%": data["main"]["humidity"],
    "Weather_Condition": data["weather"][0]["main"],
    "Wind_Speed_m_s": data["wind"]["speed"],
    "Wind direction": data["wind"]["deg"],
    "Cloudiness": data["clouds"]["all"],
    "Visibility": data["visibility"],
    "Latitude": data["coord"]["lat"],
    "Longitude": data["coord"]["lon"],
    "Date_Time": datetime.fromtimestamp(data["dt"])
}

weather_data

{'City': 'Lagos',
 'Temperature_C': 23.92,
 'Humidity_%': 90,
 'Weather_Condition': 'Clouds',
 'Wind_Speed_m_s': 2.5,
 'Wind direction': 253,
 'Cloudiness': 74,
 'Visibility': 10000,
 'Latitude': 6.5833,
 'Longitude': 3.75,
 'Date_Time': datetime.datetime(2026, 8, 18, 4, 20, 2)}

**extraction function**

In [19]:
def extract_weather(city, state, region):
    #Build the API request
    url = "https://api.openweathermap.org/data/2.5/weather"
    
    params = {
        #Sends Lagos to OpenWeather
        "q": f"{city},NG",
        #Include API key
        "appid": API_KEY,
        #Requests Celsius
        "units": "metric"
    }
    
    response = requests.get(url, params=params)
    #Checks whether the API worked
    if response.status_code != 200:
        print(f"Error retrieving {city}: {response.status_code}")
        return None
    #Converts JSON into Python
    data = response.json()
    
    weather_record = {
        "City": city,
        "State": state,
        "Region": region,
        "Country": "Nigeria",
        
        "Latitude": data["coord"]["lat"],
        "Longitude": data["coord"]["lon"],
        #Extracts the fields we care about
        "Temperature_C": data["main"]["temp"],
        "Feels_Like_C": data["main"]["feels_like"],
        "Min_Temperature_C": data["main"]["temp_min"],
        "Max_Temperature_C": data["main"]["temp_max"],
        
        "Humidity_Percent": data["main"]["humidity"],
        "Pressure_hPa": data["main"]["pressure"],
        
        "Weather_Condition": data["weather"][0]["main"],
        "Weather_Description": data["weather"][0]["description"],
        
        "Wind_Speed_mps": data["wind"]["speed"],
        "Wind_Direction_Deg": data["wind"].get("deg"),
        
        "Cloudiness_Percent": data["clouds"]["all"],
        
        "Visibility_km": data["visibility"] / 1000,
        
        "Weather_Timestamp": data["dt"],
        
        "Extracted_At": datetime.now()
    }
    
    return weather_record

In [20]:
#test the function
lagos = extract_weather(
    "Lagos",
    "Lagos",
    "South West"
)
lagos

{'City': 'Lagos',
 'State': 'Lagos',
 'Region': 'South West',
 'Country': 'Nigeria',
 'Latitude': 6.5833,
 'Longitude': 3.75,
 'Temperature_C': 27.06,
 'Feels_Like_C': 29.4,
 'Min_Temperature_C': 27.06,
 'Max_Temperature_C': 27.06,
 'Humidity_Percent': 75,
 'Pressure_hPa': 1012,
 'Weather_Condition': 'Clouds',
 'Weather_Description': 'overcast clouds',
 'Wind_Speed_mps': 2.69,
 'Wind_Direction_Deg': 224,
 'Cloudiness_Percent': 100,
 'Visibility_km': 10.0,
 'Weather_Timestamp': 1787063950,
 'Extracted_At': datetime.datetime(2026, 8, 18, 15, 41, 13, 22382)}

In [24]:
len(cities)

12

In [27]:
for location in cities:
    print(location["City"], "-", location["Region"])

Lagos - South West
Ibadan - South West
Akure - South West
Enugu - South East
Umuahia - South East
Awka - South East
Port Harcourt - South South
Calabar - South South
Abuja - North Central
Jos - North Central
Kano - North West
Maiduguri - North East


**Extract all 12 cities**

In [28]:
weather_records = []

for location in cities:
    
    weather = extract_weather(
        location["City"],
        location["State"],
        location["Region"]
    )
    
    if weather is not None:
        weather_records.append(weather)

In [29]:
#Check the number of successful responses
len(weather_records)

12

In [ ]:
#Look at the records
weather_records

[{'City': 'Lagos',
  'State': 'Lagos',
  'Region': 'South West',
  'Country': 'Nigeria',
  'Latitude': 6.5833,
  'Longitude': 3.75,
  'Temperature_C': 27.06,
  'Feels_Like_C': 29.4,
  'Min_Temperature_C': 27.06,
  'Max_Temperature_C': 27.06,
  'Humidity_Percent': 75,
  'Pressure_hPa': 1012,
  'Weather_Condition': 'Clouds',
  'Weather_Description': 'overcast clouds',
  'Wind_Speed_mps': 2.69,
  'Wind_Direction_Deg': 224,
  'Cloudiness_Percent': 100,
  'Visibility_km': 10.0,
  'Weather_Timestamp': 1787063950,
  'Extracted_At': datetime.datetime(2026, 8, 18, 15, 47, 21, 912877)},
 {'City': 'Ibadan',
  'State': 'Oyo',
  'Region': 'South West',
  'Country': 'Nigeria',
  'Latitude': 7.3878,
  'Longitude': 3.8964,
  'Temperature_C': 25.95,
  'Feels_Like_C': 25.95,
  'Min_Temperature_C': 25.95,
  'Max_Temperature_C': 25.95,
  'Humidity_Percent': 80,
  'Pressure_hPa': 1012,
  'Weather_Condition': 'Clouds',
  'Weather_Description': 'overcast clouds',
  'Wind_Speed_mps': 0.93,
  'Wind_Direction_D

In [31]:
#Convert the records into a DataFrame
df = pd.DataFrame(weather_records)
df

,City,State,Region,Country,Latitude,Longitude,Temperature_C,Feels_Like_C,Min_Temperature_C,Max_Temperature_C,Humidity_Percent,Pressure_hPa,Weather_Condition,Weather_Description,Wind_Speed_mps,Wind_Direction_Deg,Cloudiness_Percent,Visibility_km,Weather_Timestamp,Extracted_At
0,Lagos,Lagos,South West,Nigeria,6.5833,3.7500,27.06,29.40,27.06,27.06,75,1012,Clouds,overcast clouds,2.69,224,100,10.000,1787063950,2026-08-18 15:47:21.912877
1,Ibadan,Oyo,South West,Nigeria,7.3878,3.8964,25.95,25.95,25.95,25.95,80,1012,Clouds,overcast clouds,0.93,219,100,10.000,1787064406,2026-08-18 15:47:22.394393
2,Akure,Ondo,South West,Nigeria,7.2526,5.1931,24.11,24.87,24.11,24.11,88,1012,Clouds,overcast clouds,1.58,234,100,10.000,1787064187,2026-08-18 15:47:22.889826
3,Enugu,Enugu,South East,Nigeria,6.4402,7.4943,22.93,23.79,22.93,22.93,96,1012,Clouds,overcast clouds,2.15,251,100,10.000,1787064442,2026-08-18 15:47:23.405105
4,Umuahia,Abia,South East,Nigeria,5.5263,7.4896,22.85,23.70,22.85,22.85,96,1012,Rain,moderate rain,1.53,273,100,9.763,1787064443,2026-08-18 15:47:23.976765
5,Awka,Anambra,South East,Nigeria,6.2101,7.0741,23.17,24.05,23.17,23.17,96,1012,Rain,light rain,1.09,224,100,10.000,1787064443,2026-08-18 15:47:24.512658
6,Port Harcourt,Rivers,South South,Nigeria,4.7774,7.0134,23.96,24.81,23.96,23.96,92,1012,Clouds,overcast clouds,2.20,264,100,10.000,1787064174,2026-08-18 15:47:25.047557
7,Calabar,Cross River,South South,Nigeria,4.9517,8.3220,24.53,25.44,24.53,24.53,92,1012,Rain,light rain,1.87,240,99,10.000,1787064444,2026-08-18 15:47:25.611712
8,Abuja,FCT,North Central,Nigeria,9.0574,7.4898,26.65,26.65,26.65,26.65,80,1011,Clouds,overcast clouds,0.39,173,98,9.291,1787064243,2026-08-18 15:47:26.143607
9,Jos,Plateau,North Central,Nigeria,9.9167,8.9000,24.47,24.85,24.47,24.47,72,1011,Clouds,overcast clouds,4.04,102,95,10.000,1787064445,2026-08-18 15:47:26.666792


In [32]:
df.shape

(12, 20)

In [33]:
df.columns

Index(['City', 'State', 'Region', 'Country', 'Latitude', 'Longitude',
       'Temperature_C', 'Feels_Like_C', 'Min_Temperature_C',
       'Max_Temperature_C', 'Humidity_Percent', 'Pressure_hPa',
       'Weather_Condition', 'Weather_Description', 'Wind_Speed_mps',
       'Wind_Direction_Deg', 'Cloudiness_Percent', 'Visibility_km',
       'Weather_Timestamp', 'Extracted_At'],
      dtype='object')

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   City                 12 non-null     object        
 1   State                12 non-null     object        
 2   Region               12 non-null     object        
 3   Country              12 non-null     object        
 4   Latitude             12 non-null     float64       
 5   Longitude            12 non-null     float64       
 6   Temperature_C        12 non-null     float64       
 7   Feels_Like_C         12 non-null     float64       
 8   Min_Temperature_C    12 non-null     float64       
 9   Max_Temperature_C    12 non-null     float64       
 10  Humidity_Percent     12 non-null     int64         
 11  Pressure_hPa         12 non-null     int64         
 12  Weather_Condition    12 non-null     object        
 13  Weather_Description  12 non-null     

In [35]:
#Convert the weather timestamp
#Nigeria is UTC+1. But OpenWeather's dt timestamp is UTC-based.
#create a UTC timestamp first.
df["Weather_Timestamp"] = pd.to_datetime(
    df["Weather_Timestamp"],
    unit="s",
    utc=True
)


In [36]:
df[["City", "Weather_Timestamp"]]

,City,Weather_Timestamp
0,Lagos,2026-08-18 14:39:10+00:00
1,Ibadan,2026-08-18 14:46:46+00:00
2,Akure,2026-08-18 14:43:07+00:00
3,Enugu,2026-08-18 14:47:22+00:00
4,Umuahia,2026-08-18 14:47:23+00:00
5,Awka,2026-08-18 14:47:23+00:00
6,Port Harcourt,2026-08-18 14:42:54+00:00
7,Calabar,2026-08-18 14:47:24+00:00
8,Abuja,2026-08-18 14:44:03+00:00
9,Jos,2026-08-18 14:47:25+00:00


In [37]:
#Create Nigerian local time. analysis is going to be based on Nigeria time, not UTC.
df["Weather_Time_Nigeria"] = (
    df["Weather_Timestamp"]
    .dt.tz_convert("Africa/Lagos")
)
df[["City", "Weather_Timestamp", "Weather_Time_Nigeria"]]

,City,Weather_Timestamp,Weather_Time_Nigeria
0,Lagos,2026-08-18 14:39:10+00:00,2026-08-18 15:39:10+01:00
1,Ibadan,2026-08-18 14:46:46+00:00,2026-08-18 15:46:46+01:00
2,Akure,2026-08-18 14:43:07+00:00,2026-08-18 15:43:07+01:00
3,Enugu,2026-08-18 14:47:22+00:00,2026-08-18 15:47:22+01:00
4,Umuahia,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00
5,Awka,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00
6,Port Harcourt,2026-08-18 14:42:54+00:00,2026-08-18 15:42:54+01:00
7,Calabar,2026-08-18 14:47:24+00:00,2026-08-18 15:47:24+01:00
8,Abuja,2026-08-18 14:44:03+00:00,2026-08-18 15:44:03+01:00
9,Jos,2026-08-18 14:47:25+00:00,2026-08-18 15:47:25+01:00


In [38]:
#Create the extraction timestamp properly
datetime.now()
datetime.now().astimezone()

datetime.datetime(2026, 8, 18, 15, 56, 1, 709164, tzinfo=datetime.timezone(datetime.timedelta(seconds=3600), 'W. Central Africa Standard Time'))

In [39]:
#update dataframe
df["Extracted_At"] = pd.to_datetime(
    df["Extracted_At"]
)

In [40]:
#Check for missing values
df.isnull().sum()

City                    0
State                   0
Region                  0
Country                 0
Latitude                0
Longitude               0
Temperature_C           0
Feels_Like_C            0
Min_Temperature_C       0
Max_Temperature_C       0
Humidity_Percent        0
Pressure_hPa            0
Weather_Condition       0
Weather_Description     0
Wind_Speed_mps          0
Wind_Direction_Deg      0
Cloudiness_Percent      0
Visibility_km           0
Weather_Timestamp       0
Extracted_At            0
Weather_Time_Nigeria    0
dtype: int64

In [41]:
df.duplicated().sum()

np.int64(0)

In [42]:
df.describe()

,Latitude,Longitude,Temperature_C,Feels_Like_C,Min_Temperature_C,Max_Temperature_C,Humidity_Percent,Pressure_hPa,Wind_Speed_mps,Wind_Direction_Deg,Cloudiness_Percent,Visibility_km,Extracted_At
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12
mean,7.662500,7.358308,26.188333,27.129167,26.188333,26.188333,79.000000,1011.250000,1.925000,196.583333,97.083333,9.921167,2026-08-18 15:47:24.789491968
min,4.777400,3.750000,22.850000,23.700000,22.850000,22.850000,36.000000,1008.000000,0.390000,29.000000,74.000000,9.291000,2026-08-18 15:47:21.912877
25%,6.039150,6.558325,23.762500,24.620000,23.762500,23.762500,74.250000,1011.000000,1.352500,161.250000,98.750000,10.000000,2026-08-18 15:47:23.276285184
50%,6.917950,7.489700,24.500000,25.155000,24.500000,24.500000,84.000000,1012.000000,1.725000,224.000000,100.000000,10.000000,2026-08-18 15:47:24.780107520
75%,9.272225,8.370675,26.752500,27.337500,26.752500,26.752500,93.000000,1012.000000,2.322500,242.750000,100.000000,10.000000,2026-08-18 15:47:26.274403328
max,12.000100,13.160300,35.490000,36.930000,35.490000,35.490000,96.000000,1012.000000,4.040000,273.000000,100.000000,10.000000,2026-08-18 15:47:27.694045
std,2.497165,2.491003,4.056902,4.446117,4.056902,4.056902,19.904317,1.356801,1.015413,74.592783,7.415688,0.209821,NaN


**Prepare the dataset for 7 days of collection**    
12 cities × 4 observations/day × 7 days = 336 rows     
Every time the pipeline runs, we append new observations. We do not replace yesterday's data.    

In [43]:
# add a proper collection timestamp
df = df.rename(columns={
    "Extracted_At": "Extracted_At_Local"
})
df.columns

Index(['City', 'State', 'Region', 'Country', 'Latitude', 'Longitude',
       'Temperature_C', 'Feels_Like_C', 'Min_Temperature_C',
       'Max_Temperature_C', 'Humidity_Percent', 'Pressure_hPa',
       'Weather_Condition', 'Weather_Description', 'Wind_Speed_mps',
       'Wind_Direction_Deg', 'Cloudiness_Percent', 'Visibility_km',
       'Weather_Timestamp', 'Extracted_At_Local', 'Weather_Time_Nigeria'],
      dtype='object')

In [44]:
# Add a UTC extraction timestamp
df["Extracted_At_UTC"] = pd.Timestamp.now(tz="UTC")
df[[
    "City",
    "Weather_Timestamp",
    "Weather_Time_Nigeria",
    "Extracted_At_UTC"
]]

,City,Weather_Timestamp,Weather_Time_Nigeria,Extracted_At_UTC
0,Lagos,2026-08-18 14:39:10+00:00,2026-08-18 15:39:10+01:00,2026-08-18 15:06:29.784076+00:00
1,Ibadan,2026-08-18 14:46:46+00:00,2026-08-18 15:46:46+01:00,2026-08-18 15:06:29.784076+00:00
2,Akure,2026-08-18 14:43:07+00:00,2026-08-18 15:43:07+01:00,2026-08-18 15:06:29.784076+00:00
3,Enugu,2026-08-18 14:47:22+00:00,2026-08-18 15:47:22+01:00,2026-08-18 15:06:29.784076+00:00
4,Umuahia,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00,2026-08-18 15:06:29.784076+00:00
5,Awka,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00,2026-08-18 15:06:29.784076+00:00
6,Port Harcourt,2026-08-18 14:42:54+00:00,2026-08-18 15:42:54+01:00,2026-08-18 15:06:29.784076+00:00
7,Calabar,2026-08-18 14:47:24+00:00,2026-08-18 15:47:24+01:00,2026-08-18 15:06:29.784076+00:00
8,Abuja,2026-08-18 14:44:03+00:00,2026-08-18 15:44:03+01:00,2026-08-18 15:06:29.784076+00:00
9,Jos,2026-08-18 14:47:25+00:00,2026-08-18 15:47:25+01:00,2026-08-18 15:06:29.784076+00:00


In [45]:
# Add the collection date
df["Collection_Date"] = (
    df["Extracted_At_UTC"]
    .dt.date
)

# Add the collection hour
df["Collection_Hour_UTC"] = (
    df["Extracted_At_UTC"]
    .dt.hour
)

# create the Nigerian hour.
df["Collection_Time_Nigeria"] = (
    df["Extracted_At_UTC"]
    .dt.tz_convert("Africa/Lagos")
)

df["Collection_Hour_Nigeria"] = (
    df["Collection_Time_Nigeria"]
    .dt.hour
)

In [46]:
#Create the four time periods
def get_time_period(hour):
    
    if 5 <= hour < 9:
        return "Morning"
    
    elif 9 <= hour < 15:
        return "Midday"
    
    elif 15 <= hour < 21:
        return "Evening"
    
    else:
        return "Midnight"

# apply it
df["Time_Period"] = df["Collection_Hour_Nigeria"].apply(
    get_time_period
)

#check
df[[
    "City",
    "Collection_Time_Nigeria",
    "Collection_Hour_Nigeria",
    "Time_Period"
]]

,City,Collection_Time_Nigeria,Collection_Hour_Nigeria,Time_Period
0,Lagos,2026-08-18 16:06:29.784076+01:00,16,Evening
1,Ibadan,2026-08-18 16:06:29.784076+01:00,16,Evening
2,Akure,2026-08-18 16:06:29.784076+01:00,16,Evening
3,Enugu,2026-08-18 16:06:29.784076+01:00,16,Evening
4,Umuahia,2026-08-18 16:06:29.784076+01:00,16,Evening
5,Awka,2026-08-18 16:06:29.784076+01:00,16,Evening
6,Port Harcourt,2026-08-18 16:06:29.784076+01:00,16,Evening
7,Calabar,2026-08-18 16:06:29.784076+01:00,16,Evening
8,Abuja,2026-08-18 16:06:29.784076+01:00,16,Evening
9,Jos,2026-08-18 16:06:29.784076+01:00,16,Evening


In [47]:
#Create a unique observation ID

df["Observation_ID"] = (
    df["City"].str.replace(" ", "_")
    + "_"
    + df["Extracted_At_UTC"].dt.strftime("%Y%m%d_%H%M%S")
)

df[["Observation_ID", "City", "Extracted_At_UTC"]]

,Observation_ID,City,Extracted_At_UTC
0,Lagos_20260818_150629,Lagos,2026-08-18 15:06:29.784076+00:00
1,Ibadan_20260818_150629,Ibadan,2026-08-18 15:06:29.784076+00:00
2,Akure_20260818_150629,Akure,2026-08-18 15:06:29.784076+00:00
3,Enugu_20260818_150629,Enugu,2026-08-18 15:06:29.784076+00:00
4,Umuahia_20260818_150629,Umuahia,2026-08-18 15:06:29.784076+00:00
5,Awka_20260818_150629,Awka,2026-08-18 15:06:29.784076+00:00
6,Port_Harcourt_20260818_150629,Port Harcourt,2026-08-18 15:06:29.784076+00:00
7,Calabar_20260818_150629,Calabar,2026-08-18 15:06:29.784076+00:00
8,Abuja_20260818_150629,Abuja,2026-08-18 15:06:29.784076+00:00
9,Jos_20260818_150629,Jos,2026-08-18 15:06:29.784076+00:00


In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype                       
---  ------                   --------------  -----                       
 0   City                     12 non-null     object                      
 1   State                    12 non-null     object                      
 2   Region                   12 non-null     object                      
 3   Country                  12 non-null     object                      
 4   Latitude                 12 non-null     float64                     
 5   Longitude                12 non-null     float64                     
 6   Temperature_C            12 non-null     float64                     
 7   Feels_Like_C             12 non-null     float64                     
 8   Min_Temperature_C        12 non-null     float64                     
 9   Max_Temperature_C        12 non-null     float64                   

**storage**

                OPENWEATHER API
                       │
                       ▼
                   EXTRACT
                       │
                       ▼
                 Raw JSON data
                       │
                       ▼
                  TRANSFORM
                       │
            ┌──────────┼──────────┐
            ▼          ▼          ▼
          Clean      Validate   Enrich
          fields      data      metadata
            │          │          │
            └──────────┼──────────┘
                       ▼
                     LOAD
                       │
                       ▼
                   SQLite DB
                       │
                       ▼
                  SQL / Pandas
                       │
                       ▼
                   Power BI
                       │
                       ▼
                 Final Dashboard

**Set up the SQLite database**

In [52]:
#Define where the database will live
project_root = Path.cwd().parent

data_folder = project_root / "data"

data_folder.mkdir(exist_ok=True)

db_path = data_folder / "weather.db"

print(db_path)

c:\Users\ASUS\Downloads\All Data Files\weather\data\weather.db


In [53]:
#Create the SQLite database
connection = sqlite3.connect(db_path)

print("Database connection successful.")

Database connection successful.


In [54]:
#Create the cities table
create_cities_table = """
CREATE TABLE IF NOT EXISTS cities (
    city_id INTEGER PRIMARY KEY AUTOINCREMENT,
    city TEXT NOT NULL,
    state TEXT NOT NULL,
    region TEXT NOT NULL,
    country TEXT NOT NULL
);
"""

connection.execute(create_cities_table)

connection.commit()

print("Cities table created successfully.")

Cities table created successfully.


In [55]:
#Create the weather observations table
create_weather_table = """
CREATE TABLE IF NOT EXISTS weather_observations (
    observation_id TEXT PRIMARY KEY,
    
    city_id INTEGER,
    
    temperature_c REAL,
    feels_like_c REAL,
    min_temperature_c REAL,
    max_temperature_c REAL,
    
    humidity_percent INTEGER,
    pressure_hpa REAL,
    
    weather_condition TEXT,
    weather_description TEXT,
    
    wind_speed_mps REAL,
    wind_direction_deg REAL,
    
    cloudiness_percent INTEGER,
    visibility_km REAL,
    
    weather_timestamp TEXT,
    weather_time_nigeria TEXT,
    
    extracted_at_utc TEXT,
    collection_date TEXT,
    collection_hour_nigeria INTEGER,
    time_period TEXT,
    
    FOREIGN KEY (city_id) REFERENCES cities(city_id)
);
"""

connection.execute(create_weather_table)

connection.commit()

print("Weather observations table created successfully.")

Weather observations table created successfully.


In [56]:
len(cities)

12

In [57]:
insert_city = """
INSERT INTO cities (city, state, region, country)
VALUES (?, ?, ?, ?)
"""

for location in cities:
    connection.execute(
        insert_city,
        (
            location["City"],
            location["State"],
            location["Region"],
            "Nigeria"
        )
    )

connection.commit()

print("Cities inserted successfully.")

Cities inserted successfully.


In [58]:
cities_db = pd.read_sql_query(
    "SELECT * FROM cities",
    connection
)

cities_db

,city_id,city,state,region,country
0,1,Lagos,Lagos,South West,Nigeria
1,2,Ibadan,Oyo,South West,Nigeria
2,3,Akure,Ondo,South West,Nigeria
3,4,Enugu,Enugu,South East,Nigeria
4,5,Umuahia,Abia,South East,Nigeria
5,6,Awka,Anambra,South East,Nigeria
6,7,Port Harcourt,Rivers,South South,Nigeria
7,8,Calabar,Cross River,South South,Nigeria
8,9,Abuja,FCT,North Central,Nigeria
9,10,Jos,Plateau,North Central,Nigeria


In [60]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    connection
)

tables

,name
0,cities
1,sqlite_sequence
2,weather_observations


In [61]:
#connect the DataFrame to the database
city_lookup = pd.read_sql_query(
    "SELECT city_id, city FROM cities",
    connection
)

city_lookup

,city_id,city
0,1,Lagos
1,2,Ibadan
2,3,Akure
3,4,Enugu
4,5,Umuahia
5,6,Awka
6,7,Port Harcourt
7,8,Calabar
8,9,Abuja
9,10,Jos


In [62]:
# Add city_id to weather DataFrame
df = df.merge(
    city_lookup,
    left_on="City",
    right_on="city",
    how="left"
)

df[["City", "city_id"]]

,City,city_id
0,Lagos,1
1,Ibadan,2
2,Akure,3
3,Enugu,4
4,Umuahia,5
5,Awka,6
6,Port Harcourt,7
7,Calabar,8
8,Abuja,9
9,Jos,10


In [63]:
df.columns.tolist()

['City',
 'State',
 'Region',
 'Country',
 'Latitude',
 'Longitude',
 'Temperature_C',
 'Feels_Like_C',
 'Min_Temperature_C',
 'Max_Temperature_C',
 'Humidity_Percent',
 'Pressure_hPa',
 'Weather_Condition',
 'Weather_Description',
 'Wind_Speed_mps',
 'Wind_Direction_Deg',
 'Cloudiness_Percent',
 'Visibility_km',
 'Weather_Timestamp',
 'Extracted_At_Local',
 'Weather_Time_Nigeria',
 'Extracted_At_UTC',
 'Collection_Date',
 'Collection_Hour_UTC',
 'Collection_Time_Nigeria',
 'Collection_Hour_Nigeria',
 'Time_Period',
 'Observation_ID',
 'city_id',
 'city']

In [64]:
# remove duplicate city column
df = df.drop(columns=["city"])
df.columns.tolist()

['City',
 'State',
 'Region',
 'Country',
 'Latitude',
 'Longitude',
 'Temperature_C',
 'Feels_Like_C',
 'Min_Temperature_C',
 'Max_Temperature_C',
 'Humidity_Percent',
 'Pressure_hPa',
 'Weather_Condition',
 'Weather_Description',
 'Wind_Speed_mps',
 'Wind_Direction_Deg',
 'Cloudiness_Percent',
 'Visibility_km',
 'Weather_Timestamp',
 'Extracted_At_Local',
 'Weather_Time_Nigeria',
 'Extracted_At_UTC',
 'Collection_Date',
 'Collection_Hour_UTC',
 'Collection_Time_Nigeria',
 'Collection_Hour_Nigeria',
 'Time_Period',
 'Observation_ID',
 'city_id']

In [66]:
load_df = df[[
    "Observation_ID",
    "city_id",
    "Temperature_C",
    "Feels_Like_C",
    "Min_Temperature_C",
    "Max_Temperature_C",
    "Humidity_Percent",
    "Pressure_hPa",
    "Weather_Condition",
    "Weather_Description",
    "Wind_Speed_mps",
    "Wind_Direction_Deg",
    "Cloudiness_Percent",
    "Visibility_km",
    "Weather_Timestamp",
    "Weather_Time_Nigeria",
    "Extracted_At_UTC",
    "Collection_Date",
    "Collection_Hour_Nigeria",
    "Time_Period"
]].copy()

# rename
load_df = load_df.rename(columns={
    "Observation_ID": "observation_id",
    "Temperature_C": "temperature_c",
    "Feels_Like_C": "feels_like_c",
    "Min_Temperature_C": "min_temperature_c",
    "Max_Temperature_C": "max_temperature_c",
    "Humidity_Percent": "humidity_percent",
    "Pressure_hPa": "pressure_hpa",
    "Weather_Condition": "weather_condition",
    "Weather_Description": "weather_description",
    "Wind_Speed_mps": "wind_speed_mps",
    "Wind_Direction_Deg": "wind_direction_deg",
    "Cloudiness_Percent": "cloudiness_percent",
    "Visibility_km": "visibility_km",
    "Weather_Timestamp": "weather_timestamp",
    "Weather_Time_Nigeria": "weather_time_nigeria",
    "Extracted_At_UTC": "extracted_at_utc",
    "Collection_Date": "collection_date",
    "Collection_Hour_Nigeria": "collection_hour_nigeria",
    "Time_Period": "time_period"
})

In [67]:
load_df.head()

,observation_id,city_id,temperature_c,feels_like_c,min_temperature_c,max_temperature_c,humidity_percent,pressure_hpa,weather_condition,weather_description,wind_speed_mps,wind_direction_deg,cloudiness_percent,visibility_km,weather_timestamp,weather_time_nigeria,extracted_at_utc,collection_date,collection_hour_nigeria,time_period
0,Lagos_20260818_150629,1,27.06,29.40,27.06,27.06,75,1012,Clouds,overcast clouds,2.69,224,100,10.000,2026-08-18 14:39:10+00:00,2026-08-18 15:39:10+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
1,Ibadan_20260818_150629,2,25.95,25.95,25.95,25.95,80,1012,Clouds,overcast clouds,0.93,219,100,10.000,2026-08-18 14:46:46+00:00,2026-08-18 15:46:46+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
2,Akure_20260818_150629,3,24.11,24.87,24.11,24.11,88,1012,Clouds,overcast clouds,1.58,234,100,10.000,2026-08-18 14:43:07+00:00,2026-08-18 15:43:07+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
3,Enugu_20260818_150629,4,22.93,23.79,22.93,22.93,96,1012,Clouds,overcast clouds,2.15,251,100,10.000,2026-08-18 14:47:22+00:00,2026-08-18 15:47:22+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
4,Umuahia_20260818_150629,5,22.85,23.70,22.85,22.85,96,1012,Rain,moderate rain,1.53,273,100,9.763,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening


In [68]:
# Load the weather observations
load_df.to_sql(
    "weather_observations",
    connection,
    if_exists="append",
    index=False
)

print("Weather observations loaded successfully.")
#verify database
weather_db = pd.read_sql_query(
    "SELECT * FROM weather_observations",
    connection
)

weather_db

Weather observations loaded successfully.


,observation_id,city_id,temperature_c,feels_like_c,min_temperature_c,max_temperature_c,humidity_percent,pressure_hpa,weather_condition,weather_description,wind_speed_mps,wind_direction_deg,cloudiness_percent,visibility_km,weather_timestamp,weather_time_nigeria,extracted_at_utc,collection_date,collection_hour_nigeria,time_period
0,Lagos_20260818_150629,1,27.06,29.40,27.06,27.06,75,1012.0,Clouds,overcast clouds,2.69,224.0,100,10.000,2026-08-18 14:39:10+00:00,2026-08-18 15:39:10+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
1,Ibadan_20260818_150629,2,25.95,25.95,25.95,25.95,80,1012.0,Clouds,overcast clouds,0.93,219.0,100,10.000,2026-08-18 14:46:46+00:00,2026-08-18 15:46:46+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
2,Akure_20260818_150629,3,24.11,24.87,24.11,24.11,88,1012.0,Clouds,overcast clouds,1.58,234.0,100,10.000,2026-08-18 14:43:07+00:00,2026-08-18 15:43:07+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
3,Enugu_20260818_150629,4,22.93,23.79,22.93,22.93,96,1012.0,Clouds,overcast clouds,2.15,251.0,100,10.000,2026-08-18 14:47:22+00:00,2026-08-18 15:47:22+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
4,Umuahia_20260818_150629,5,22.85,23.70,22.85,22.85,96,1012.0,Rain,moderate rain,1.53,273.0,100,9.763,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
5,Awka_20260818_150629,6,23.17,24.05,23.17,23.17,96,1012.0,Rain,light rain,1.09,224.0,100,10.000,2026-08-18 14:47:23+00:00,2026-08-18 15:47:23+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
6,Port_Harcourt_20260818_150629,7,23.96,24.81,23.96,23.96,92,1012.0,Clouds,overcast clouds,2.20,264.0,100,10.000,2026-08-18 14:42:54+00:00,2026-08-18 15:42:54+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
7,Calabar_20260818_150629,8,24.53,25.44,24.53,24.53,92,1012.0,Rain,light rain,1.87,240.0,99,10.000,2026-08-18 14:47:24+00:00,2026-08-18 15:47:24+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
8,Abuja_20260818_150629,9,26.65,26.65,26.65,26.65,80,1011.0,Clouds,overcast clouds,0.39,173.0,98,9.291,2026-08-18 14:44:03+00:00,2026-08-18 15:44:03+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening
9,Jos_20260818_150629,10,24.47,24.85,24.47,24.47,72,1011.0,Clouds,overcast clouds,4.04,102.0,95,10.000,2026-08-18 14:47:25+00:00,2026-08-18 15:47:25+01:00,2026-08-18 15:06:29.784076+00:00,2026-08-18,16,Evening


In [69]:
weather_db[["observation_id", "city_id", "temperature_c", "humidity_percent"]]

,observation_id,city_id,temperature_c,humidity_percent
0,Lagos_20260818_150629,1,27.06,75
1,Ibadan_20260818_150629,2,25.95,80
2,Akure_20260818_150629,3,24.11,88
3,Enugu_20260818_150629,4,22.93,96
4,Umuahia_20260818_150629,5,22.85,96
5,Awka_20260818_150629,6,23.17,96
6,Port_Harcourt_20260818_150629,7,23.96,92
7,Calabar_20260818_150629,8,24.53,92
8,Abuja_20260818_150629,9,26.65,80
9,Jos_20260818_150629,10,24.47,72


In [72]:
# Test the relationship
query = """
SELECT
    w.observation_id,
    c.city,
    c.state,
    c.region,
    w.temperature_c,
    w.humidity_percent,
    w.weather_condition,
    w.wind_speed_mps
FROM weather_observations AS w
JOIN cities AS c
    ON w.city_id = c.city_id
"""

weather_analysis = pd.read_sql_query(
    query,
    connection
)

weather_analysis

,observation_id,city,state,region,temperature_c,humidity_percent,weather_condition,wind_speed_mps
0,Lagos_20260818_150629,Lagos,Lagos,South West,27.06,75,Clouds,2.69
1,Ibadan_20260818_150629,Ibadan,Oyo,South West,25.95,80,Clouds,0.93
2,Akure_20260818_150629,Akure,Ondo,South West,24.11,88,Clouds,1.58
3,Enugu_20260818_150629,Enugu,Enugu,South East,22.93,96,Clouds,2.15
4,Umuahia_20260818_150629,Umuahia,Abia,South East,22.85,96,Rain,1.53
5,Awka_20260818_150629,Awka,Anambra,South East,23.17,96,Rain,1.09
6,Port_Harcourt_20260818_150629,Port Harcourt,Rivers,South South,23.96,92,Clouds,2.20
7,Calabar_20260818_150629,Calabar,Cross River,South South,24.53,92,Rain,1.87
8,Abuja_20260818_150629,Abuja,FCT,North Central,26.65,80,Clouds,0.39
9,Jos_20260818_150629,Jos,Plateau,North Central,24.47,72,Clouds,4.04


In [ ]:
#connection.close()